In [170]:
import pandas as pd
from tqdm.std import tqdm
from openai import OpenAI
import re
import os
import json

In [171]:
input_data = "input_files/semantic_matching_0123.csv"
output_data = "output_files/semantic_matching_0123.xlsx"

In [ ]:
## Open AI API
os.environ["OPENAI_API_KEY"] = "your open ai api"

## common functions

In [173]:
def to_pascal_case(phrase):
    # Remove parentheses and the content inside them, e.g., (Unaudited)
    phrase = re.sub(r'\s*\([^)]*\)', '', phrase)
    
    # Split on non-alphanumeric characters
    words = re.split(r'[^a-zA-Z0-9]', phrase)
    
    # Capitalize each word and join
    return ''.join(word.capitalize() for word in words if word)

In [186]:
def read_xml(xml_path):
    with open(xml_path, "r", encoding="utf-8") as f:
        content = f.read()
        return content

In [193]:
"""
    This function is for the semantic matching task.
"""
def construct_annotation(df, results, task_name, base_data_path, rule_id="0123"):
    """
        the options of task_name are semantic_matching, relationship_extraction, and mathematical_reasoning
    """
    new_df = df.copy(deep=True)

    schema = []
    presentation = []
    calculation = []
    definition = []
    label = []
    instance = []
    us_gaap = []
    true_answer = []
    segmentation_path = []
    parse_ok = []
    input_col = []
    outputByLLM = []

    for res in results:
        folder_name = res.get("folder_name")
        prefix_path = os.path.join(base_data_path, folder_name)
        
        parse = False
        cal, pre, defn, lab, ins, sch, seg_path = [None] * 7
        
        ## special for semantic matching task (DQC.US.0123)
        flattened_answers = []
        if res.get("main_element"):
            flattened_answers.append(res.get("main_element"))
        
        error_dimensions = res.get("error_dimensions", [])
        if error_dimensions and isinstance(error_dimensions, list):
            for err_dim in error_dimensions:
                if err_dim and err_dim.get("dimension"):
                    flattened_answers.extend(err_dim.get("dimension").split('='))
        
        answers = res.get("error_dimensions") 

        if os.path.exists(prefix_path) and os.path.isdir(prefix_path):
            all_subfolders = [d for d in os.listdir(prefix_path) if os.path.isdir(os.path.join(prefix_path, d))]

            for subfolder_name in all_subfolders:
                full_path = os.path.join(prefix_path, subfolder_name)
                temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch = [None] * 6
                
                file_list = os.listdir(full_path)
                for file_name in file_list:
                    final_path = os.path.join(full_path, file_name)
                    if file_name.endswith("cal.xml"): temp_cal = read_xml(final_path)
                    elif file_name.endswith("pre.xml"): temp_pre = read_xml(final_path)
                    elif file_name.endswith("def.xml"): defn = read_xml(final_path)
                    elif file_name.endswith("lab.xml"): lab = read_xml(final_path)
                    elif file_name.endswith("htm.xml"): temp_ins = read_xml(final_path)
                    elif file_name.endswith("xsd"): sch = read_xml(final_path)
                
                is_appear = True
                ## check the instance document
                instance_content = temp_ins or ""
                
                if not flattened_answers or not instance_content:
                    is_appear = False
                else:
                    for ans in flattened_answers:
                        if not (ans in instance_content or ans.replace(":", "_") in instance_content):
                            is_appear = False
                            break
               
                if is_appear:
                    parse = True
                    cal, pre, defn, lab, ins, sch = temp_cal, temp_pre, temp_defn, temp_lab, temp_ins, temp_sch
                    seg_path = os.path.join(folder_name, subfolder_name)
                    break 
        
        calculation.append(cal)
        presentation.append(pre)
        definition.append(defn)
        label.append(lab)
        instance.append(ins)
        schema.append(sch)
        true_answer.append(answers)
        us_gaap.append(None)
        parse_ok.append(parse)
        segmentation_path.append(seg_path)
        input_col.append(None)
        outputByLLM.append(res)
     
    new_df["schema"] = schema
    new_df["presentation"] = presentation
    new_df["calculation"] = calculation
    new_df["definition"] = definition
    new_df["label"] = label
    new_df["instance"] = instance
    new_df["us_gaap"] = us_gaap
    new_df["input"] = input_col
    new_df["true_answer"] = true_answer
    new_df["segmentation_path"] = segmentation_path
    new_df["parse_successful"] = parse_ok
    new_df["LLM_Output"] = outputByLLM

    return new_df

## load dqc messages for semantic matching task (DQC.US.0123)

In [176]:
sm_123_df = pd.read_csv(input_data)

In [177]:
sm_123_df.head()

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,report.base-taxonomy,report.creation-software,report.document-type,report.entry-url,report.filing-date,report.filing-year,report.id,report.sec-url,report.sic-code,folder_name
0,DQC.US.0123.9583,"2022-02-09 10:14:50,180 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,US GAAP 2021,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1580262...,2022-02-09,2022,417122,https://www.sec.gov/Archives/edgar/data/158026...,7374,10q-rton-20211231
1,DQC.US.0123.9583,"2022-02-09 10:14:50,180 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,US GAAP 2021,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1580262...,2022-02-09,2022,417122,https://www.sec.gov/Archives/edgar/data/158026...,7374,10q-rton-20211231
2,DQC.US.0123.9583,"2022-02-09 10:14:50,181 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,US GAAP 2021,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1580262...,2022-02-09,2022,417122,https://www.sec.gov/Archives/edgar/data/158026...,7374,10q-rton-20211231
3,DQC.US.0123.9583,"2022-02-09 10:14:50,181 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,US GAAP 2021,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1580262...,2022-02-09,2022,417122,https://www.sec.gov/Archives/edgar/data/158026...,7374,10q-rton-20211231
4,DQC.US.0123.9583,"2022-02-09 10:14:50,181 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,US GAAP 2021,Document Created by XBRLMaster,10-Q,http://www.sec.gov/Archives/edgar/data/1580262...,2022-02-09,2022,417122,https://www.sec.gov/Archives/edgar/data/158026...,7374,10q-rton-20211231


## LLM judge assisstant

In [ ]:
Client = OpenAI()

In [179]:
sm_123_system_prompt = """You are an expert in XBRL taxonomies, tasked with parsing DQC validation messages to identify errors related to incomplete dimensional structures (Rule DQC.US.0123). Your goal is to extract four specific pieces of information:

1. **Statement Name**: Extract the clean name of the financial statement where the error occurred (e.g., "Statement of Shareholders Equity").

2. **Main Element**: Extract the primary `us-gaap:` concept being discussed. This is the central fact of the error report and is explicitly labeled on a line starting with 'Element :'.

3. **Erroneous Dimension**: Identify the specific dimension string from the message that is part of the incomplete structure. An entry should be included **only if** it satisfies the following conditions:
    * It must be the full string explicitly listed in the **'Dimensions'** field of the message.
    * You must capture this string **exactly as it appears**, preserving all prefixes (e.g., `us-gaap:`, `rton:`).
    * The message must describe this dimension as being part of an incomplete fact.

4. **Reasoning**: For the extracted dimension, provide a brief explanation summarizing why it is erroneous, typically because it requires an additional axis as described in the message.

Output your result as a single, structured JSON object in the following format:

```json
{
  "statement_name": "...",
  "main_element": "us-gaap:StockholdersEquity",
  "error_dimensions": [
    {
      "dimension": "us-gaap:StatementClassOfStockAxis=rton:PreferredStocksMember",
      "reason": "The message states this dimension is incomplete for the main element because it is missing another required axis."
    }
  ]
}
```"""

In [180]:
def get_response(dqc_message):
    
    completion = Client.chat.completions.create(
        model="gpt-4.1-mini",
        response_format={
            "type": "json_object"
        },
        messages=[
            {"role": "system", "content": sm_123_system_prompt},
            {"role": "user", "content": f"DQC validation message: {dqc_message} \n Output:"}
        ]
    )
    return completion.choices[0].message.content

In [ ]:
res = []

# to run on the full dataset, use the line below:
for _, row in tqdm(sm_123_df.iterrows(), total=sm_123_df.shape[0]):
# to test on only the first 10 examples, comment out the line above and uncomment the line below:
# for _, row in tqdm(sm_123_df.head(10).iterrows(), total=10):
    
    # load each dqc message
    message = row["assertion.detail"]
    response = get_response(message)
    response = json.loads(response)

    # load the folder name which will be used to obtain candidate segmentation folders
    folder_name = row["folder_name"]

    # covert the extracted statement name in the message to PascalCase format as the segmentation name
    segmentation_name = to_pascal_case(response.get("statement_name", ""))

    # record the folder name
    response["folder_name"] = folder_name
    # record the extracted segmenataion name
    response["segmentation_name"] = segmentation_name
    res.append(response)

In [188]:
# Display the first result to verify
res[0]

{'statement_name': 'Statement of Shareholders Equity',
 'main_element': 'us-gaap:StockholdersEquity',
 'error_dimensions': [{'dimension': 'us-gaap:StatementClassOfStockAxis=rton:PreferredStocksMember',
   'reason': 'The message states this dimension is incomplete for the main element because it is missing the required Statement Equity Components Axis with the appropriate stock member.'}],
 'folder_name': '10q-rton-20211231',
 'segmentation_name': 'StatementOfShareholdersEquity'}

In [194]:
DATA_FOLDER_PATH = "./segmentation_data/semantic_matching"

new_sm_123_df = construct_annotation(sm_123_df, res, 
                                     task_name="semantic_matching", 
                                     base_data_path=DATA_FOLDER_PATH)

In [191]:
new_sm_123_df

,assertion.code,assertion.detail,assertion.run-date,assertion.severity,assertion.source,assertion.type,entity.cik,entity.name,report.accepted-timestamp,report.accession,...,calculation,definition,label,instance,us_gaap,input,true_answer,segmentation_path,parse_successful,LLM_Output
0,DQC.US.0123.9583,"2022-02-09 10:14:50,180 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[{'dimension': 'us-gaap:StatementClassOfStockA...,10q-rton-20211231/CondensedConsolidatedBalance...,True,{'statement_name': 'Statement of Shareholders ...
1,DQC.US.0123.9583,"2022-02-09 10:14:50,180 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[{'dimension': 'us-gaap:StatementClassOfStockA...,10q-rton-20211231/CondensedConsolidatedBalance...,True,{'statement_name': 'Statement of Shareholders ...
2,DQC.US.0123.9583,"2022-02-09 10:14:50,181 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[{'dimension': 'us-gaap:StatementClassOfStockA...,10q-rton-20211231/CondensedConsolidatedBalance...,True,{'statement_name': 'Statement of Shareholders ...
3,DQC.US.0123.9583,"2022-02-09 10:14:50,181 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[{'dimension': 'us-gaap:StatementClassOfStockA...,10q-rton-20211231/CondensedConsolidatedBalance...,True,{'statement_name': 'Statement of Shareholders ...
4,DQC.US.0123.9583,"2022-02-09 10:14:50,181 [DQC.US.0123.9583] The...",2022-02-09,ERROR,DQC,123,1580262,"Right On Brands, Inc.",2022-02-09 09:00:00,0001477932-22-000634,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[{'dimension': 'us-gaap:StatementClassOfStockA...,10q-rton-20211231/CondensedConsolidatedBalance...,True,{'statement_name': 'Statement of Shareholders ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
160,DQC.US.0123.9583,"2024-11-18 01:08:56,142 [DQC.US.0123.9583] The...",2024-11-18,ERROR,DQC,123,1919246,Channel Therapeutics Corporation,2024-11-13 16:30:00,0001753926-24-001883,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,None,[{'dimension': 'us-gaap:StatementClassOfStockA...,10q-chro-20240930/CondensedConsolidatedBalance...,True,{'statement_name': 'Statement of Shareholders ...
161,DQC.US.0123.9583,"2024-11-18 01:08:56,142 [DQC.US.0123.9583] The...",2024-11-18,ERROR,DQC,123,1919246,Channel Therapeutics Corporation,2024-11-13 16:30:00,0001753926-24-001883,...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='US-ASCII'?>\n<li...,<?xml version='1.0' encoding='UTF-8'?>\n<link:...,<?xml version='1.0' encoding='UTF-8'?>\n<xbrl ...,None,N

In [196]:
new_sm_123_df.to_excel(output_data, index=False)